# Paula's Choice Ingredient Data — Cleaning Pipeline

**Input:** `paula_choice_full_details.json` (raw scraped data)  
**Output:** `paula_choice_cleaned.json` (knowledge base ready for RAG embedding)

### Pipeline Steps
1. Load JSON → raw DataFrame + quick inspect  
2. Cleaning (normalize, drop, dedup, flag)  
3. Add `safety_label` + `concern_group` + `warnings`  
4. Add `rag_text` field  
5. Data quality report  
6. Export → cleaned JSON

## Cell 1 — Imports + Config

In [ ]:
import json
import re
import pandas as pd
from collections import defaultdict

# ── Paths ──────────────────────────────────────────────────────────────
INPUT_PATH  = 'paula_choice_full_details.json'
OUTPUT_PATH = 'paula_choice_cleaned.json'

# ── Safety mapping (Paula's Choice rating → numeric) ───────────────────
# Used INTERNALLY for concern_group classification only
# Never exposed as an aggregate score in the UI
SAFETY_MAP = {
    'Best':    5,
    'Good':    4,
    'Average': 3,
    'Bad':     2,
    'Worst':   1,
}

SAFETY_LABEL = {
    5: 'Very Safe',
    4: 'Safe',
    3: 'Use with Awareness',
    2: 'Caution Advised',
    1: 'Known Concern',
}

# ── Concern group thresholds ────────────────────────────────────────────
# Category override takes priority over score
OVERRIDE_TO_CONCERN        = {'Irritant'}
OVERRIDE_TO_WORTH_KNOWING  = {'Preservative', 'Fragrance: Synthetic and Natural'}

# ── Warning messages per category ───────────────────────────────────────
WARNING_TRIGGERS = {
    'Irritant':                        'May cause irritation — patch test recommended.',
    'Fragrance: Synthetic and Natural': 'Common allergen — avoid if sensitive skin.',
    'Preservative':                    'Preservative — considered safe within regulated limits.',
}

print('Config loaded')
print(f'  SAFETY_MAP              : {SAFETY_MAP}')
print(f'  Override → concern      : {OVERRIDE_TO_CONCERN}')
print(f'  Override → worth_knowing: {OVERRIDE_TO_WORTH_KNOWING}')

## Cell 2 — Load JSON → Raw DataFrame + Quick Inspect

In [ ]:
with open(INPUT_PATH, encoding='utf-8') as f:
    raw = json.load(f)

df_raw = pd.DataFrame(raw)

print(f'Shape   : {df_raw.shape}')
print(f'Columns : {list(df_raw.columns)}')
print(f'\nRating distribution:')
print(df_raw['rating'].value_counts().to_string())
print(f'\nMissing values per column:')
missing = df_raw.isnull().sum()
print(missing[missing > 0].to_string())
print(f'\nEmpty description : {(df_raw["description"].fillna("") == "").sum()}')
print(f'Empty key_points  : {(df_raw["key_points"].apply(lambda x: len(x) == 0)).sum()}')
print(f'Empty benefits    : {(df_raw["benefits"].apply(lambda x: len(x) == 0)).sum()}')
print(f'Empty references  : {(df_raw["references"].apply(lambda x: len(x) == 0)).sum()}')

df_raw.head(2)

## Cell 3 — Cleaning

- Normalize title (strip whitespace, collapse spaces)
- Clean unicode artifacts in description
- Rename fields to project-standard naming
- Drop records with no description AND no key_points
- Flag records missing description only (keep, has key_points)
- Deduplicate — keep longest description, union all list fields

In [ ]:
def normalize_title(title: str) -> str:
    return re.sub(r'\s+', ' ', str(title)).strip()

def clean_text(text: str) -> str:
    if not text:
        return ''
    replacements = {
        '\u00a0': ' ',   # non-breaking space
        '\u202f': ' ',   # narrow no-break space (22 descriptions affected)
        '\u2009': ' ',   # thin space
        '\u200b': '',    # zero-width space
        '\u2013': '-',   # en-dash
        '\u2014': '-',   # em-dash
        '\u2019': "'",   # right single quote
        '\u201c': '"',   # left double quote
        '\u201d': '"',   # right double quote
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return re.sub(r'\s+', ' ', text).strip()

def clean_list(lst) -> list:
    if not isinstance(lst, list):
        return []
    return [str(x).strip() for x in lst if str(x).strip()]

# ── Step 1: Rename + clean fields ───────────────────────────────────────
df = pd.DataFrame({
    'name':                df_raw['h1_title'].apply(normalize_title),
    'rating':              df_raw['rating'].fillna(''),
    'rating_value':        df_raw['rating_value'],
    'description':         df_raw['description'].fillna('').apply(clean_text),
    'key_points':          df_raw['key_points'].apply(clean_list),
    'benefits':            df_raw['benefits'].apply(clean_list),
    'categories':          df_raw['categories'].apply(clean_list),
    'related_ingredients': df_raw['related_ingredients'].apply(clean_list),
    'references':          df_raw['references'].apply(clean_list),
    'date_modified':       df_raw['date_modified'].fillna(''),
    'source_url':          df_raw['url'].fillna(''),
})

before = len(df)

# ── Step 2: Drop records with no description AND no key_points ──────────
no_content_mask = (df['description'] == '') & (df['key_points'].apply(len) == 0)
dropped = df[no_content_mask]['name'].tolist()
df = df[~no_content_mask].copy()
print(f'[DROP] {before - len(df)} records with no content:')
print(f'       {dropped}')

# ── Step 3: Flag records missing description only ───────────────────────
df['description_missing'] = df['description'] == ''
flagged = df[df['description_missing']]['name'].tolist()
print(f'\n[FLAG] {len(flagged)} records missing description (kept, have key_points):')
print(f'       {flagged}')

# ── Step 4: Deduplicate — keep longest description, union list fields ────
LIST_FIELDS = ['key_points', 'benefits', 'categories', 'related_ingredients', 'references']

def merge_group(group):
    best = group.loc[group['description'].str.len().idxmax()].copy()
    for field in LIST_FIELDS:
        merged, seen = [], set()
        for row_items in group[field]:
            for item in row_items:
                if item.lower() not in seen:
                    seen.add(item.lower())
                    merged.append(item)
        best[field] = merged
    return best

before_dedup = len(df)

# Track which records were merged (for audit)
dedup_log = {}
for key, group in df.groupby(df['name'].str.lower(), sort=False):
    if len(group) > 1:
        best_idx = group['description'].str.len().idxmax()
        dedup_log[key] = {
            'kept':    df.loc[best_idx, 'name'],
            'count':   len(group),
            'desc_lens': group['description'].str.len().tolist(),
        }

df = (
    df.groupby(df['name'].str.lower(), sort=False)
      .apply(merge_group)
      .reset_index(drop=True)
)
print(f'\n[DEDUP] {before_dedup - len(df)} duplicate records merged → {len(df)} unique')
print(f'\nDuplicate groups found:')
for key, info in dedup_log.items():
    print(f"  {info['kept']:45s} | {info['count']} records | desc_lens={info['desc_lens']}")

print(f'\nCleaned shape: {df.shape}')
df.head(3)

## Cell 4 — Add `safety_label` + `concern_group` + `warnings`

Classification logic (see `STRATEGY.md`):
- Category override takes **priority** over score
- `Irritant` → always `potential_concern`
- `Preservative` / `Fragrance: Synthetic and Natural` → always `worth_knowing`
- Score fallback: 4–5 → no_concern · 3 → worth_knowing · 1–2 → potential_concern

In [ ]:
def get_concern_group(row) -> str:
    score      = SAFETY_MAP.get(row['rating'], 0)
    categories = set(row['categories'])
    # Score <= 2 always potential_concern — override cannot downgrade a bad rating
    if score <= 2:
        return 'potential_concern'
    # Category override — takes priority for score >= 3
    if categories & OVERRIDE_TO_CONCERN:
        return 'potential_concern'
    if categories & OVERRIDE_TO_WORTH_KNOWING:
        return 'worth_knowing'
    # Score-based fallback
    if score >= 4:
        return 'no_concern'
    else:  # score == 3
        return 'worth_knowing'

def get_warnings(row) -> list:
    return [
        f"{row['name']}: {WARNING_TRIGGERS[cat]}"
        for cat in row['categories']
        if cat in WARNING_TRIGGERS
    ]

df['safety_score']  = df['rating'].map(SAFETY_MAP).fillna(0).astype(int)
df['safety_label']  = df['safety_score'].map(SAFETY_LABEL).fillna('Unknown')
df['concern_group'] = df.apply(get_concern_group, axis=1)
df['warnings']      = df.apply(get_warnings, axis=1)

print('Concern group distribution:')
print(df['concern_group'].value_counts().to_string())

print('\nSafety label distribution:')
print(df['safety_label'].value_counts().to_string())

# Spot checks
for name in ['Fragrance', 'Phenoxyethanol', 'Niacinamide']:
    row = df[df['name'].str.lower() == name.lower()]
    if not row.empty:
        r = row.iloc[0]
        print(f'\n--- Spot check: {name} ---')
        print(f'  rating       : {r["rating"]}')
        print(f'  concern_group: {r["concern_group"]}')
        print(f'  warnings     : {r["warnings"]}')

## Cell 5 — Add `rag_text` Field

Single text block per ingredient for embedding.  
Format ensures semantic search hits on **name, categories, key_points AND description**.

In [ ]:
GROUP_LABEL_MAP = {
    'no_concern':       'No Concerns',
    'worth_knowing':    'Worth Knowing',
    'potential_concern':'Potential Concern',
}

def build_rag_text(row) -> str:
    parts = [f"Ingredient: {row['name']}"]
    if row['categories']:
        parts.append('Categories: ' + ', '.join(row['categories']))
    if row['key_points']:
        parts.append('Key Points: ' + '. '.join(row['key_points']))
    if row['description']:
        parts.append(row['description'])
    if row['benefits']:
        parts.append('Benefits: ' + ', '.join(row['benefits']))
    if row['safety_label']:
        parts.append(f"Safety: {row['safety_label']}")
    if row['concern_group']:
        parts.append(f"Concern Group: {GROUP_LABEL_MAP.get(row['concern_group'], '')}")
    return ' | '.join(parts)

df['rag_text'] = df.apply(build_rag_text, axis=1)

rag_lengths = df['rag_text'].str.len()
print('rag_text length stats:')
print(f'  min  : {rag_lengths.min()} chars')
print(f'  max  : {rag_lengths.max()} chars')
print(f'  mean : {rag_lengths.mean():.0f} chars')
print(f'  >500 chars : {(rag_lengths > 500).sum()} records')
print(f'  <100 chars : {(rag_lengths < 100).sum()} records  <- short, review if needed')

print('\n--- Sample rag_text (Niacinamide) ---')
nia = df[df['name'].str.lower() == 'niacinamide']
if not nia.empty:
    print(nia.iloc[0]['rag_text'][:500] + '...')

## Cell 6 — Data Quality Report

In [ ]:
print('=' * 55)
print('DATA QUALITY REPORT')
print('=' * 55)

print(f'\nTotal records          : {len(df)}')
print(f'description_missing    : {df["description_missing"].sum()}')
print(f'Empty benefits         : {(df["benefits"].apply(len) == 0).sum()}')
print(f'Empty references       : {(df["references"].apply(len) == 0).sum()}')
print(f'Empty categories       : {(df["categories"].apply(len) == 0).sum()}')

print('\n── Concern Group Distribution ──')
for group, count in df['concern_group'].value_counts().items():
    pct = count / len(df) * 100
    print(f'  {group:22s}: {count:4d}  ({pct:.1f}%)')

print('\n── Top 15 Categories ──')
all_cats = [cat for cats in df['categories'] for cat in cats]
for cat, count in pd.Series(all_cats).value_counts().head(15).items():
    print(f'  {cat:45s}: {count}')

print(f'\n── Records with Warnings ──')
has_warn = df['warnings'].apply(len) > 0
print(f'  {has_warn.sum()} records have at least one warning')

print('\n── Final Columns ──')
print(df.columns.tolist())

## Cell 7 — Export → `paula_choice_cleaned.json`

In [ ]:
EXPORT_COLS = [
    'name',
    'rating',
    'safety_score',
    'safety_label',
    'concern_group',
    'description',
    'description_missing',
    'key_points',
    'benefits',
    'categories',
    'related_ingredients',
    'warnings',
    'references',
    'date_modified',
    'source_url',
    'rag_text',
]

df_export = df[EXPORT_COLS].sort_values('name', key=lambda s: s.str.lower())
records = df_export.to_dict(orient='records')

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

import os
print(f'Exported {len(records)} records → {OUTPUT_PATH}')
print(f'File size: {os.path.getsize(OUTPUT_PATH) / 1024:.1f} KB')

# Final preview
print('\n--- Sample record (Niacinamide) ---')
sample = next(r for r in records if r['name'].lower() == 'niacinamide')
for k, v in sample.items():
    if k == 'rag_text':
        print(f'  rag_text   : {str(v)[:120]}...')
    elif k == 'references':
        print(f'  references : [{len(v)} entries]')
    else:
        print(f'  {k:22s}: {v}')